# Bank Customer Churn Analysis — Data Cleaning

## 1. Objective

Clean and prepare the Customer_Info and Account_Info datasets for accurate customer churn analysis.

### Data Cleaning Tasks
- Identify and remove duplicate records.
- Handle missing values.
- Standardize inconsistent categorical values.
- Correct data types.
- Validate customer identifiers and relationships.
- Export cleaned datasets for PostgreSQL and Power BI.

## 2. Duplicate Records Identification

Duplicate customer records can inflate customer counts and distort churn calculations. We first identify exact duplicate rows and verify their contents before removing them.

In [13]:
import pandas as pd
from pathlib import Path

project_path = Path(r"E:\BA Projects\Bank_Customer_Churn_Analysis")
raw_path = project_path / "Data" / "Raw"

customer_df = pd.read_excel(raw_path / "customer_info.xlsx")
account_df = pd.read_excel(raw_path / "account_info.xlsx")

print("CUSTOMER INFO - DUPLICATE RECORDS")
print(customer_df[customer_df.duplicated(keep=False)])

print("ACCOUNT INFO - DUPLICATE RECORDS")
print(account_df[account_df.duplicated(keep=False)])


CUSTOMER INFO - DUPLICATE RECORDS
       CustomerId Surname  CreditScore Geography  Gender   Age  Tenure  \
9999     15628319  Walker          792    French  Female  28.0       4   
10000    15628319  Walker          792    French  Female  28.0       4   

      EstimatedSalary  
9999        €38190.78  
10000       €38190.78  
ACCOUNT INFO - DUPLICATE RECORDS
       CustomerId     Balance  NumOfProducts HasCrCard  Tenure IsActiveMember  \
0        15634602        €0.0              1       Yes       2            Yes   
1        15634602        €0.0              1       Yes       2            Yes   
10000    15628319  €130142.79              1        No       4             No   
10001    15628319  €130142.79              1        No       4             No   

       Exited  
0           1  
1           1  
10000       0  
10001       0  


### Duplicate Records — Findings

- Customer_Info contains 10,001 rows, including 1 exact duplicate of CustomerId 15628319.
- Account_Info contains 10,002 rows, including 2 extra duplicate rows associated with CustomerIds 15634602 and 15628319.
- All identified duplicate records are identical across their respective columns.
- Decision: Retain the first occurrence of each exact duplicate and remove subsequent copies.
- Expected result: 10,000 records in each dataset after deduplication.

In [7]:

# Create working copies to preserve raw data
customer_clean = customer_df.copy()
account_clean = account_df.copy()

# Remove exact duplicate rows
customer_clean = customer_clean.drop_duplicates(
    keep="first"
).reset_index(drop=True)

account_clean = account_clean.drop_duplicates(
    keep="first"
).reset_index(drop=True)

print("Customer_Info:", customer_clean.shape)
print("Account_Info:", account_clean.shape)


Customer_Info: (10000, 8)
Account_Info: (10000, 7)


In [8]:

print("CUSTOMER INFO")
print("Duplicate rows:",
      customer_clean.duplicated().sum())
print("Duplicate CustomerIds:",
      customer_clean["CustomerId"].duplicated().sum())

print("\nACCOUNT INFO")
print("Duplicate rows:",
      account_clean.duplicated().sum())
print("Duplicate CustomerIds:",
      account_clean["CustomerId"].duplicated().sum())

assert len(customer_clean) == 10000
assert len(account_clean) == 10000
assert customer_clean["CustomerId"].is_unique
assert account_clean["CustomerId"].is_unique

print("\nDuplicate cleaning validation passed!")


CUSTOMER INFO
Duplicate rows: 0
Duplicate CustomerIds: 0

ACCOUNT INFO
Duplicate rows: 0
Duplicate CustomerIds: 0

Duplicate cleaning validation passed!


### Duplicate Cleaning Summary

- Removed 1 exact duplicate from Customer_Info.
- Removed 2 exact duplicates from Account_Info.
- Both datasets now contain 10,000 unique customer records.
- CustomerId uniqueness was validated in both datasets.
- Original raw Excel files remain unchanged.

**Result:** Duplicate cleaning completed successfully.

## 3. Missing Value Analysis

### Objective
Identify missing values in the cleaned customer and account datasets and determine appropriate treatment methods without unnecessarily losing customer records.

### Approach
- Calculate missing values for each column.
- Inspect records containing missing information.
- Evaluate whether missing values should be retained, imputed, or removed.
- Validate the results after treatment.

**Business importance:** Missing demographic information can affect customer segmentation and churn analysis. Any treatment must preserve data integrity and minimize analytical bias.

In [12]:
# Count missing values after duplicate removal
print("CUSTOMER INFO - MISSING VALUES")
print(customer_clean.isna().sum())

print("\nACCOUNT INFO - MISSING VALUES")
print(account_clean.isna().sum())

# Display customers with missing Surname or Age
missing_records = customer_clean[customer_clean[["Surname","Age"]].isna().any(axis=1)]

print("\nCUSTOMERS WITH MISSING VALUES")
display(missing_records)

print("Total affected customer records:", len(missing_records))

CUSTOMER INFO - MISSING VALUES
CustomerId         0
Surname            3
CreditScore        0
Geography          0
Gender             0
Age                3
Tenure             0
EstimatedSalary    0
dtype: int64

ACCOUNT INFO - MISSING VALUES
CustomerId        0
Balance           0
NumOfProducts     0
HasCrCard         0
Tenure            0
IsActiveMember    0
Exited            0
dtype: int64

CUSTOMERS WITH MISSING VALUES


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,EstimatedSalary
28,15728693,NaN,574,Germany,Female,NaN,3,-€999999
121,15580203,NaN,674,Spain,Male,NaN,6,-€999999
9389,15756954,NaN,538,France,Female,NaN,2,-€999999


Total affected customer records: 3


### Missing Value Investigation — Findings

Three customer records contain missing values in both `Surname` and `Age`. These same records also contain an invalid `EstimatedSalary` value of `-€999999`.

**Affected Customer IDs:**
- 15728693
- 15580203
- 15756954

**Cleaning decision:** Before applying imputation or deleting records, investigate whether the original values can be recovered from the clean reference dataset.

In [14]:

# Load the clean reference dataset
reference_df = pd.read_excel(
    raw_path / "Bank_Churn.xlsx"
)

# Customer IDs requiring investigation
missing_ids = [15728693, 15580203, 15756954]

# Find these customers in the reference dataset
reference_records = reference_df[
    reference_df["CustomerId"].isin(missing_ids)
]

print("Reference dataset columns:")
print(reference_df.columns.tolist())

print("\nMatching reference records:")
display(reference_records)

print("\nNumber of matching records:",
      len(reference_records))


Reference dataset columns:
['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']

Matching reference records:


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
28,15728693,McWilliams,574,Germany,Female,43,3,141349.43,1,1,1,100187.43,0
121,15580203,Kennedy,674,Spain,Male,39,6,120193.42,1,0,0,100130.95,0
9389,15756954,Lombardo,538,France,Female,32,2,0.00,1,1,1,80130.54,0



Number of matching records: 3


### 3.2 Handling Missing Values

**Problem:** Three customer records have missing Surname and Age values, along with invalid EstimatedSalary values of -€999999.

**Solution:** Recover the original values from Bank_Churn.xlsx using CustomerId instead of deleting records or filling missing values with estimates.

**Objective:** Restore accurate customer information while preserving all 10,000 customer records.

In [15]:

# Restore missing values using reference dataset

missing_ids = [15728693, 15580203, 15756954]

reference_lookup = reference_df.set_index("CustomerId")

for customer_id in missing_ids:

    mask = customer_clean["CustomerId"] == customer_id

    customer_clean.loc[mask, "Surname"] = (
        reference_lookup.loc[customer_id, "Surname"]
    )

    customer_clean.loc[mask, "Age"] = (
        reference_lookup.loc[customer_id, "Age"]
    )

    customer_clean.loc[mask, "EstimatedSalary"] = (
        "€" + str(reference_lookup.loc[
            customer_id, "EstimatedSalary"
        ])
    )

# Display corrected records
display(
    customer_clean[
        customer_clean["CustomerId"].isin(missing_ids)
    ]
)


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,EstimatedSalary
28,15728693,McWilliams,574,Germany,Female,43.0,3,€100187.43
121,15580203,Kennedy,674,Spain,Male,39.0,6,€100130.95
9389,15756954,Lombardo,538,France,Female,32.0,2,€80130.54


In [16]:

# Validate missing value recovery

print("Missing values after cleaning:")
print(customer_clean.isnull().sum())

print("\nTotal customer records:", len(customer_clean))

print("Duplicate CustomerIds:",
      customer_clean["CustomerId"].duplicated().sum())

# Validation checks
assert customer_clean.isnull().sum().sum() == 0
assert len(customer_clean) == 10000
assert customer_clean["CustomerId"].is_unique

print("\nMissing value validation passed!")


Missing values after cleaning:
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
EstimatedSalary    0
dtype: int64

Total customer records: 10000
Duplicate CustomerIds: 0

Missing value validation passed!


### 3.2 Missing Value Cleaning — Summary

- Identified 3 customer records with missing Surname and Age values.
- Recovered the original values using the clean reference dataset and CustomerId.
- Corrected 3 invalid EstimatedSalary placeholders.
- Preserved all 10,000 customer records.
- Confirmed there are no remaining missing values or duplicate CustomerIds.

**Result:** Missing value cleaning completed successfully.

### 3.3 Standardizing Geography Values

**Objective:** Identify inconsistent country names in the Geography column and standardize them into a consistent format.

**Business importance:** Inconsistent country names can split the same country into multiple categories, resulting in inaccurate customer counts and churn rates by geography.

**Approach:**
1. Identify unique Geography values.
2. Determine which values represent the same country.
3. Standardize country names.
4. Validate the cleaned values.

In [18]:
print("Unique Geography Values:")
print(customer_clean["Geography"].unique())

print("\nGeography Value Counts:")
print(customer_clean["Geography"].value_counts(dropna=False))

Unique Geography Values:
<StringArray>
['FRA', 'Spain', 'French', 'France', 'Germany', 'Yes']
Length: 6, dtype: str

Geography Value Counts:
Geography
Germany    2509
Spain      2476
France     1741
French     1655
FRA        1618
Yes           1
Name: count, dtype: int64


In [20]:

# Investigate invalid Geography value

invalid_geo = customer_clean[
    customer_clean["Geography"] == "Yes"
]

print("Invalid Geography Record:")
display(invalid_geo)

# Check the original country using reference data
invalid_ids = invalid_geo["CustomerId"]

reference_geo = reference_df[
    reference_df["CustomerId"].isin(invalid_ids)
]

print("\nOriginal Geography from Reference Dataset:")
display(
    reference_geo[["CustomerId", "Geography"]]
)


Invalid Geography Record:


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,EstimatedSalary
21,15597945,Dellucci,636,Yes,Female,32.0,8,€138555.46



Original Geography from Reference Dataset:


,CustomerId,Geography
21,15597945,Spain


In [21]:

# Standardize Geography values

customer_clean["Geography"] = (
    customer_clean["Geography"]
    .replace({
        "FRA": "France",
        "French": "France"
    })
)

# Restore invalid Geography using reference dataset
customer_clean.loc[
    customer_clean["CustomerId"] == 15597945,
    "Geography"
] = "Spain"

# Verify cleaned values
print("Geography after cleaning:")
print(customer_clean["Geography"].value_counts())

# Validation
assert set(customer_clean["Geography"].unique()) == {
    "France", "Germany", "Spain"
}

assert len(customer_clean) == 10000

print("\nGeography cleaning validation passed!")


Geography after cleaning:
Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

Geography cleaning validation passed!


### 3.4 Gender Standardization

**Objective:** Identify and standardize inconsistent values in the Gender column.

**Business importance:** Consistent gender categories are necessary for accurate customer segmentation and churn rate comparisons.

**Approach:**
1. Identify unique Gender values.
2. Check for inconsistent or invalid entries.
3. Standardize values where necessary.
4. Validate the cleaned categories.

In [31]:

# Check unique Gender values

print("Unique Gender Values:")
print(customer_clean["Gender"].unique())

print("\nMissing Gender Values:")
print(customer_clean["Gender"].isna().sum())


print("\nGender Value Counts:")
print(customer_clean["Gender"].value_counts(dropna=False))

print("\n Total Gender Value Counts:")
print(customer_clean["Gender"].value_counts(dropna=False).sum())



Unique Gender Values:
<StringArray>
['Female', 'Male']
Length: 2, dtype: str

Missing Gender Values:
0

Gender Value Counts:
Gender
Male      5457
Female    4543
Name: count, dtype: int64

 Total Gender Value Counts:
10000


In [32]:

# Validate Gender values

assert set(customer_clean["Gender"].unique()) == {
    "Male", "Female"
}

assert customer_clean["Gender"].notna().all()

assert len(customer_clean) == 10000

print("Gender validation passed!")


Gender validation passed!


### 3.4 Gender Standardization — Summary

**Findings:**
- The Gender column contains two categories: Male and Female.
- Male customers: 5,457.
- Female customers: 4,543.
- No missing or inconsistent values were identified.

**Cleaning decision:** No transformation was required because the existing values were already standardized.

**Result:** Gender validation completed successfully.

In [34]:

# Inspect EstimatedSalary before cleaning

print("Current Data Type:")
print(customer_clean["EstimatedSalary"].dtype)

print("\nFirst 10 Salary Values:")
print(customer_clean["EstimatedSalary"].head(10))

print("\nMissing Salary Values:")
print(customer_clean["EstimatedSalary"].isna().sum())

# Identify values that cannot be converted after
# removing the euro symbol and thousands separators
salary_text = (
    customer_clean["EstimatedSalary"]
    .astype("string")
    .str.replace("€", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

invalid_salary = pd.to_numeric(salary_text, errors="coerce").isna()

print("\nInvalid Salary Values:")
print(invalid_salary.sum())

display(
    customer_clean.loc[
        invalid_salary,
        ["CustomerId", "EstimatedSalary"]
    ]
)


Current Data Type:
str

First 10 Salary Values:
0    €101348.88
1    €112542.58
2    €113931.57
3     €93826.63
4      €79084.1
5    €149756.71
6      €10062.8
7    €119346.88
8      €74940.5
9     €71725.73
Name: EstimatedSalary, dtype: str

Missing Salary Values:
0

Invalid Salary Values:
0


,CustomerId,EstimatedSalary


In [35]:

# Convert EstimatedSalary from string to numeric

customer_clean["EstimatedSalary"] = pd.to_numeric(
    salary_text,
    errors="raise"
)

# Verify data type
print("Data Type After Conversion:")
print(customer_clean["EstimatedSalary"].dtype)

# Check salary statistics
print("\nSalary Statistics:")
print(customer_clean["EstimatedSalary"].describe())

# Check negative salary values
print("\nNegative Salary Count:")
print((customer_clean["EstimatedSalary"] < 0).sum())

# Validate conversion
assert customer_clean["EstimatedSalary"].notna().all()
assert pd.api.types.is_numeric_dtype(
    customer_clean["EstimatedSalary"]
)

print("\nSalary conversion validation passed!")


Data Type After Conversion:
Float64

Salary Statistics:
count          10000.0
mean     100090.239881
std       57510.492818
min              11.58
25%           51002.11
50%         100193.915
75%        149388.2475
max          199992.48
Name: EstimatedSalary, dtype: Float64

Negative Salary Count:
0

Salary conversion validation passed!


### 3.5 EstimatedSalary Cleaning — Summary

**Issues identified:**
- EstimatedSalary was stored as a string.
- Salary values contained the euro (€) currency symbol.
- Numeric calculations could not be performed directly on the original values.

**Cleaning performed:**
- Removed currency symbols and formatting characters.
- Converted EstimatedSalary to a numeric Float64 data type.
- Preserved all 10,000 customer records.

**Validation results:**
- Missing values: 0
- Negative salaries: 0
- Minimum salary: €11.58
- Average salary: €100,090.24
- Median salary: €100,193.92
- Maximum salary: €199,992.48

**Result:** EstimatedSalary is ready for statistical analysis, SQL queries, and Power BI reporting.

### 3.6 CreditScore Validation

**Objective:** Validate the accuracy and consistency of customer credit scores.

**Business importance:** CreditScore is a useful variable for customer segmentation and churn analysis. Invalid scores could distort statistical calculations and dashboard insights.

**Validation approach:**
1. Check the current data type.
2. Identify missing values.
3. Review descriptive statistics.
4. Identify scores outside the expected range of 300–850.
5. Determine whether any corrections are required.

In [36]:

# Inspect CreditScore column

print("CreditScore Data Type:")
print(customer_clean["CreditScore"].dtype)

print("\nMissing CreditScore Values:")
print(customer_clean["CreditScore"].isna().sum())

print("\nCreditScore Statistics:")
print(customer_clean["CreditScore"].describe())

# Identify scores outside the expected range
invalid_credit = customer_clean[
    (customer_clean["CreditScore"] < 300) |
    (customer_clean["CreditScore"] > 850)
]

print("\nInvalid CreditScore Count:")
print(len(invalid_credit))

print("\nInvalid CreditScore Records:")
display(
    invalid_credit[["CustomerId", "CreditScore"]]
)


CreditScore Data Type:
int64

Missing CreditScore Values:
0

CreditScore Statistics:
count    10000.000000
mean       650.528800
std         96.653299
min        350.000000
25%        584.000000
50%        652.000000
75%        718.000000
max        850.000000
Name: CreditScore, dtype: float64

Invalid CreditScore Count:
0

Invalid CreditScore Records:


,CustomerId,CreditScore


### 3.6 CreditScore Validation — Summary

**Findings:**
- CreditScore contains 10,000 valid numeric records.
- Data type: int64.
- No missing values were identified.
- No scores were outside the expected range of 300–850.

**Descriptive statistics:**
- Minimum: 350
- Average: 650.53
- Median: 652
- Maximum: 850

**Cleaning decision:** No transformation was required because all values passed validation.

**Result:** CreditScore is ready for customer segmentation and churn analysis.

### 3.7 Age Validation

**Objective:** Ensure the Age column contains valid numeric values suitable for customer segmentation and churn analysis.

**Business importance:** Age helps identify customer groups with higher churn rates and supports age-based analysis in Python, SQL, and Power BI.

**Validation approach:**
1. Inspect the current data type.
2. Check for missing values.
3. Review minimum, maximum, and average age.
4. Identify potentially unrealistic ages.
5. Determine whether data type conversion or corrections are required.

In [37]:

# Inspect Age column

print("Age Data Type:")
print(customer_clean["Age"].dtype)

print("\nMissing Age Values:")
print(customer_clean["Age"].isna().sum())

print("\nAge Statistics:")
print(customer_clean["Age"].describe())

# Identify ages outside the expected range
invalid_age = customer_clean[
    (customer_clean["Age"] < 18) |
    (customer_clean["Age"] > 100)
]

print("\nPotentially Invalid Age Count:")
print(len(invalid_age))

print("\nPotentially Invalid Age Records:")
display(
    invalid_age[["CustomerId", "Age"]]
)


Age Data Type:
float64

Missing Age Values:
0

Age Statistics:
count    10000.000000
mean        38.921800
std         10.487806
min         18.000000
25%         32.000000
50%         37.000000
75%         44.000000
max         92.000000
Name: Age, dtype: float64

Potentially Invalid Age Count:
0

Potentially Invalid Age Records:


,CustomerId,Age


In [39]:

# Check whether all ages are whole numbers

fractional_age_count = (
    (customer_clean["Age"] % 1) != 0
).sum()

print("Fractional Age Count:", fractional_age_count)

# Convert Age to integer only if all values are whole numbers
assert fractional_age_count == 0, "Some ages contain decimals"

customer_clean["Age"] = customer_clean["Age"].astype("int64")

# Final validation
assert customer_clean["Age"].notna().all()
assert customer_clean["Age"].between(18, 100).all()
assert pd.api.types.is_integer_dtype(customer_clean["Age"])

print("\nAge Data Type After Conversion:")
print(customer_clean["Age"].dtype)

print("\nAge validation passed!")


Fractional Age Count: 0

Age Data Type After Conversion:
int64

Age validation passed!


### 3.7 Age Validation — Summary

**Findings:**
- The Age column contains 10,000 records.
- No missing values were identified.
- All ages fall within the expected range of 18–100.
- Minimum age: 18 years.
- Average age: 38.92 years.
- Maximum age: 92 years.

**Cleaning performed:**
- Verified that all ages are whole numbers.
- Converted Age from float64 to int64.

**Result:** Age is ready for demographic segmentation and churn analysis.

### 3.8 Tenure Validation

**Objective:** Validate customer tenure values and check consistency between the customer and account datasets.

**Business importance:** Tenure helps analyze customer loyalty and identify whether churn rates differ between new and long-term customers.

**Validation approach:**
1. Inspect the Tenure data type in both datasets.
2. Check for missing values.
3. Review minimum, maximum, and average tenure.
4. Identify potentially invalid tenure values.
5. Compare tenure values across both datasets using CustomerId.

In [40]:

# Inspect Tenure in both datasets

print("Customer Tenure Data Type:")
print(customer_clean["Tenure"].dtype)

print("\nAccount Tenure Data Type:")
print(account_clean["Tenure"].dtype)

print("\nMissing Tenure Values:")
print("Customer:", customer_clean["Tenure"].isna().sum())
print("Account:", account_clean["Tenure"].isna().sum())

print("\nCustomer Tenure Statistics:")
print(customer_clean["Tenure"].describe())

print("\nAccount Tenure Statistics:")
print(account_clean["Tenure"].describe())

# Inspect values outside the expected 0–10 year range
print("\nCustomer Tenure Outside 0–10:")
print((~customer_clean["Tenure"].between(0, 10)).sum())

print("\nAccount Tenure Outside 0–10:")
print((~account_clean["Tenure"].between(0, 10)).sum())


Customer Tenure Data Type:
int64

Account Tenure Data Type:
int64

Missing Tenure Values:
Customer: 0
Account: 0

Customer Tenure Statistics:
count    10000.000000
mean         5.012800
std          2.892174
min          0.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         10.000000
Name: Tenure, dtype: float64

Account Tenure Statistics:
count    10000.000000
mean         5.012800
std          2.892174
min          0.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         10.000000
Name: Tenure, dtype: float64

Customer Tenure Outside 0–10:
0

Account Tenure Outside 0–10:
0


In [41]:

# Compare Tenure between customer and account tables

tenure_check = customer_clean[
    ["CustomerId", "Tenure"]
].merge(
    account_clean[["CustomerId", "Tenure"]],
    on="CustomerId",
    how="outer",
    suffixes=("_customer", "_account"),
    indicator=True,
    validate="one_to_one"
)

# Check unmatched CustomerIds
print("CustomerId Match Results:")
print(tenure_check["_merge"].value_counts())

# Identify customers with different Tenure values
tenure_mismatch = tenure_check[
    (tenure_check["_merge"] == "both") &
    (
        tenure_check["Tenure_customer"] !=
        tenure_check["Tenure_account"]
    )
]

print("\nTenure Mismatch Count:")
print(len(tenure_mismatch))

print("\nTenure Mismatch Records:")
display(tenure_mismatch.head(10))


CustomerId Match Results:
_merge
both          10000
left_only         0
right_only        0
Name: count, dtype: int64

Tenure Mismatch Count:
0

Tenure Mismatch Records:


,CustomerId,Tenure_customer,Tenure_account,_merge


### 3.8 Tenure Validation — Summary

**Findings:**
- Both customer and account datasets contain 10,000 records.
- Tenure is stored as int64 in both datasets.
- No missing or out-of-range tenure values were identified.
- Tenure ranges from 0 to 10 years.
- Average customer tenure is approximately 5.01 years.

**Cross-table consistency:**
- All 10,000 CustomerIds matched successfully.
- No unmatched CustomerIds were found.
- No tenure mismatches were identified.

**Cleaning decision:**
- No changes to Tenure values are required.
- Retain only one Tenure column when merging the datasets to avoid redundant fields.

**Result:** Tenure validation completed successfully.

### 3.9 Balance Validation and Data Type Conversion

**Objective:** Ensure the Balance column contains valid numeric values suitable for churn analysis.

**Business importance:** Account balance helps identify whether customers with low, medium, or high balances have different churn rates.

**Validation approach:**
1. Inspect the Balance data type.
2. Check for missing values.
3. Review sample balance values.
4. Identify invalid numeric formats.
5. Convert Balance to a numeric data type and validate the results.

In [42]:

# Inspect Balance column

print("Balance Data Type:")
print(account_clean["Balance"].dtype)

print("\nFirst 10 Balance Values:")
print(account_clean["Balance"].head(10))

print("\nMissing Balance Values:")
print(account_clean["Balance"].isna().sum())

# Prepare Balance values for numeric conversion
balance_text = (
    account_clean["Balance"]
    .astype("string")
    .str.replace("€", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# Identify values that cannot be converted to numbers
invalid_balance = pd.to_numeric(
    balance_text,
    errors="coerce"
).isna()

print("\nInvalid Balance Values:")
print(invalid_balance.sum())

print("\nInvalid Balance Records:")
display(
    account_clean.loc[
        invalid_balance,
        ["CustomerId", "Balance"]
    ]
)


Balance Data Type:
str

First 10 Balance Values:
0          €0.0
1     €83807.86
2     €159660.8
3          €0.0
4    €125510.82
5    €113755.78
6          €0.0
7    €115046.74
8    €142051.07
9    €134603.88
Name: Balance, dtype: str

Missing Balance Values:
0

Invalid Balance Values:
0

Invalid Balance Records:


,CustomerId,Balance


In [43]:

# Convert Balance from string to numeric

account_clean["Balance"] = pd.to_numeric(
    balance_text,
    errors="raise"
)

# Check data type after conversion
print("Balance Data Type After Conversion:")
print(account_clean["Balance"].dtype)

# Review balance statistics
print("\nBalance Statistics:")
print(account_clean["Balance"].describe())

# Check zero balances
print("\nZero Balance Count:")
print((account_clean["Balance"] == 0).sum())

# Check negative balances
print("\nNegative Balance Count:")
print((account_clean["Balance"] < 0).sum())

# Validate numeric conversion
assert account_clean["Balance"].notna().all()

assert pd.api.types.is_numeric_dtype(
    account_clean["Balance"]
)

print("\nBalance conversion validation passed!")


Balance Data Type After Conversion:
Float64

Balance Statistics:
count         10000.0
mean     76485.889288
std      62397.405202
min               0.0
25%               0.0
50%          97198.54
75%         127644.24
max         250898.09
Name: Balance, dtype: Float64

Zero Balance Count:
3617

Negative Balance Count:
0

Balance conversion validation passed!


### 3.9 Balance Cleaning — Summary

**Issues identified:**
- Balance was stored as a string.
- Values contained the euro (€) currency symbol.
- Numeric conversion was required for financial analysis.

**Cleaning performed:**
- Removed currency symbols and formatting characters.
- Converted Balance to the numeric Float64 data type.
- Preserved all 10,000 account records.

**Validation results:**
- Missing values: 0
- Invalid numeric formats: 0
- Negative balances: 0
- Zero-balance customers: 3,617 (36.17%)
- Average balance: €76,485.89
- Median balance: €97,198.54
- Maximum balance: €250,898.09

**Cleaning decision:** Retained zero-balance records because zero is a valid account balance.

**Result:** Balance is ready for SQL analysis, Python EDA, and Power BI reporting.

### 3.10 NumOfProducts Validation

**Objective:** Validate the number of banking products held by each customer.

**Business importance:** Product ownership is an important factor in customer retention analysis. Understanding the relationship between product count and churn can help identify opportunities for cross-selling and customer engagement.

**Validation approach:**
1. Inspect the NumOfProducts data type.
2. Check for missing values.
3. Review the distribution of product counts.
4. Identify zero, negative, or non-integer values.
5. Determine whether any cleaning is required.

In [44]:

# Inspect NumOfProducts column

print("NumOfProducts Data Type:")
print(account_clean["NumOfProducts"].dtype)

print("\nMissing Values:")
print(account_clean["NumOfProducts"].isna().sum())

print("\nProduct Count Distribution:")
print(
    account_clean["NumOfProducts"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nNumOfProducts Statistics:")
print(account_clean["NumOfProducts"].describe())

# Identify invalid product counts
product_numeric = pd.to_numeric(
    account_clean["NumOfProducts"],
    errors="coerce"
)

invalid_products = account_clean[
    product_numeric.isna() |
    (product_numeric <= 0) |
    ((product_numeric % 1) != 0)
]

print("\nPotentially Invalid Product Count:")
print(len(invalid_products))

display(
    invalid_products[["CustomerId", "NumOfProducts"]]
)


NumOfProducts Data Type:
int64

Missing Values:
0

Product Count Distribution:
NumOfProducts
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64

NumOfProducts Statistics:
count    10000.000000
mean         1.530200
std          0.581654
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          4.000000
Name: NumOfProducts, dtype: float64

Potentially Invalid Product Count:
0


,CustomerId,NumOfProducts


### 3.10 NumOfProducts Validation — Summary

**Findings:**
- NumOfProducts contains 10,000 records.
- Data type: int64.
- No missing, zero, negative, or fractional product counts were identified.
- Product counts range from 1 to 4.

**Product distribution:**
- 1 product: 5,084 customers (50.84%)
- 2 products: 4,590 customers (45.90%)
- 3 products: 266 customers (2.66%)
- 4 products: 60 customers (0.60%)

**Cleaning decision:** No transformation was required because all values passed validation.

**Business observation:** Most customers hold one or two banking products. The relationship between product ownership and churn will be investigated during exploratory data analysis.

**Result:** NumOfProducts validation completed successfully.

### 3.11 HasCrCard Validation

**Objective:** Validate and standardize the credit-card ownership indicator.

**Business importance:** Credit-card ownership is an important customer attribute that may help explain differences in churn behavior.

**Validation approach:**
1. Inspect the HasCrCard data type.
2. Check for missing values.
3. Identify unique values and their distribution.
4. Detect inconsistent or invalid entries.
5. Standardize the column into a consistent binary format (1 = Yes, 0 = No), if necessary.

In [45]:

# Inspect HasCrCard column

print("HasCrCard Data Type:")
print(account_clean["HasCrCard"].dtype)

print("\nMissing Values:")
print(account_clean["HasCrCard"].isna().sum())

print("\nUnique Values:")
print(account_clean["HasCrCard"].unique())

print("\nValue Distribution:")
print(
    account_clean["HasCrCard"]
    .value_counts(dropna=False)
)

# Check total records
print("\nTotal Records:")
print(len(account_clean))


HasCrCard Data Type:
str

Missing Values:
0

Unique Values:
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

Value Distribution:
HasCrCard
Yes    5151
No     4849
Name: count, dtype: int64

Total Records:
10000


In [46]:

# Convert HasCrCard from Yes/No to 1/0

account_clean["HasCrCard"] = (
    account_clean["HasCrCard"]
    .map({
        "Yes": 1,
        "No": 0
    })
    .astype("int64")
)

# Check converted values
print("HasCrCard Data Type After Conversion:")
print(account_clean["HasCrCard"].dtype)

print("\nHasCrCard Distribution:")
print(account_clean["HasCrCard"].value_counts())

# Final validation
assert account_clean["HasCrCard"].notna().all()

assert set(account_clean["HasCrCard"].unique()) == {0, 1}

assert len(account_clean) == 10000

print("\nHasCrCard conversion validation passed!")


HasCrCard Data Type After Conversion:
int64

HasCrCard Distribution:
HasCrCard
1    5151
0    4849
Name: count, dtype: int64

HasCrCard conversion validation passed!


### 3.11 HasCrCard Cleaning — Summary

**Findings:**
- The HasCrCard column contained two valid categories: Yes and No.
- No missing or invalid values were identified.
- 5,151 customers (51.51%) have a credit card.
- 4,849 customers (48.49%) do not have a credit card.

**Cleaning performed:**
- Converted Yes to 1.
- Converted No to 0.
- Changed the column data type from string to int64.

**Validation:** Confirmed that all 10,000 records contain only 0 or 1.

**Result:** HasCrCard is standardized and ready for customer churn analysis.

### 3.12 IsActiveMember Validation

**Objective:** Validate and standardize the customer activity indicator.

**Business importance:** Customer activity may be associated with retention and churn. Comparing churn rates between active and inactive customers can help identify potential retention opportunities.

**Validation approach:**
1. Inspect the current data type.
2. Check for missing values.
3. Identify unique values and their distribution.
4. Detect unexpected categories.
5. Convert the values into a consistent binary format if necessary.

In [47]:

# Inspect IsActiveMember column

print("IsActiveMember Data Type:")
print(account_clean["IsActiveMember"].dtype)

print("\nMissing Values:")
print(account_clean["IsActiveMember"].isna().sum())

print("\nUnique Values:")
print(account_clean["IsActiveMember"].unique())

print("\nValue Distribution:")
print(
    account_clean["IsActiveMember"]
    .value_counts(dropna=False)
)


IsActiveMember Data Type:
str

Missing Values:
0

Unique Values:
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

Value Distribution:
IsActiveMember
Yes    5151
No     4849
Name: count, dtype: int64


In [48]:

# Compare HasCrCard and IsActiveMember

# Convert activity values temporarily for comparison
activity_numeric = account_clean["IsActiveMember"].map({
    "Yes": 1,
    "No": 0
})

print("HasCrCard vs IsActiveMember:")
print(
    pd.crosstab(
        account_clean["HasCrCard"],
        activity_numeric,
        rownames=["HasCrCard"],
        colnames=["IsActiveMember"]
    )
)

print("\nNumber of matching records:")
print((account_clean["HasCrCard"] == activity_numeric).sum())

print("\nNumber of different records:")
print((account_clean["HasCrCard"] != activity_numeric).sum())


HasCrCard vs IsActiveMember:
IsActiveMember     0     1
HasCrCard                 
0               4849     0
1                  0  5151

Number of matching records:
10000

Number of different records:
0


In [49]:

# Compare IsActiveMember with clean reference dataset

activity_check = account_clean[
    ["CustomerId", "IsActiveMember"]
].merge(
    reference_df[["CustomerId", "IsActiveMember"]],
    on="CustomerId",
    how="left",
    suffixes=("_account", "_reference"),
    validate="one_to_one"
)

print("Reference IsActiveMember Distribution:")
print(
    activity_check["IsActiveMember_reference"]
    .value_counts(dropna=False)
)

print("\nFirst 10 Records:")
display(activity_check.head(10))

print("\nMissing Reference Matches:")
print(
    activity_check["IsActiveMember_reference"]
    .isna().sum()
)


Reference IsActiveMember Distribution:
IsActiveMember_reference
1    5151
0    4849
Name: count, dtype: int64

First 10 Records:


,CustomerId,IsActiveMember_account,IsActiveMember_reference
0,15634602,Yes,1
1,15647311,Yes,1
2,15619304,No,0
3,15701354,No,0
4,15737888,Yes,1
5,15574012,No,0
6,15592531,Yes,1
7,15656148,No,0
8,15792365,Yes,1
9,15592389,Yes,1



Missing Reference Matches:
0


In [50]:

# Convert account activity values temporarily
activity_check["Activity_Account_Numeric"] = (
    activity_check["IsActiveMember_account"]
    .map({"Yes": 1, "No": 0})
)

# Compare account values against reference values
activity_mismatch = activity_check[
    activity_check["Activity_Account_Numeric"] !=
    activity_check["IsActiveMember_reference"]
]

print("Total Records Compared:", len(activity_check))

print("Missing Reference Matches:")
print(activity_check["IsActiveMember_reference"].isna().sum())

print("\nActivity Mismatch Count:")
print(len(activity_mismatch))

print("\nSample Mismatched Records:")
display(activity_mismatch.head(10))


Total Records Compared: 10000
Missing Reference Matches:
0

Activity Mismatch Count:
0

Sample Mismatched Records:


,CustomerId,IsActiveMember_account,IsActiveMember_reference,Activity_Account_Numeric


In [51]:

# Convert IsActiveMember from Yes/No to 1/0

account_clean["IsActiveMember"] = (
    account_clean["IsActiveMember"]
    .map({
        "Yes": 1,
        "No": 0
    })
    .astype("int64")
)

# Check results
print("Data Type:")
print(account_clean["IsActiveMember"].dtype)

print("\nActivity Distribution:")
print(account_clean["IsActiveMember"].value_counts())

# Validate conversion
assert account_clean["IsActiveMember"].notna().all()

assert set(account_clean["IsActiveMember"].unique()) == {0, 1}

assert len(account_clean) == 10000

print("\nIsActiveMember conversion validation passed!")


Data Type:
int64

Activity Distribution:
IsActiveMember
1    5151
0    4849
Name: count, dtype: int64

IsActiveMember conversion validation passed!


### 3.12 IsActiveMember Cleaning — Summary

**Findings:**
- IsActiveMember contained two categories: Yes and No.
- No missing or unexpected values were identified.
- All 10,000 activity values matched the clean reference dataset.

**Cleaning performed:**
- Converted Yes to 1 (Active).
- Converted No to 0 (Inactive).
- Changed the data type from string to int64.

**Validation results:**
- Active customers: 5,151 (51.51%)
- Inactive customers: 4,849 (48.49%)
- Missing values: 0
- Reference mismatches: 0

**Result:** IsActiveMember is standardized and ready for customer churn analysis.

In [52]:

# Compare HasCrCard with the reference dataset

card_check = account_clean[
    ["CustomerId", "HasCrCard"]
].merge(
    reference_df[["CustomerId", "HasCrCard"]],
    on="CustomerId",
    how="left",
    suffixes=("_account", "_reference"),
    validate="one_to_one"
)

# Identify mismatched values
card_mismatch = card_check[
    card_check["HasCrCard_account"] !=
    card_check["HasCrCard_reference"]
]

print("Total Records Compared:", len(card_check))

print("\nReference HasCrCard Distribution:")
print(card_check["HasCrCard_reference"].value_counts(dropna=False))

print("\nMissing Reference Matches:")
print(card_check["HasCrCard_reference"].isna().sum())

print("\nHasCrCard Mismatch Count:")
print(len(card_mismatch))

print("\nSample Mismatched Records:")
display(card_mismatch.head(10))


Total Records Compared: 10000

Reference HasCrCard Distribution:
HasCrCard_reference
1    7055
0    2945
Name: count, dtype: int64

Missing Reference Matches:
0

HasCrCard Mismatch Count:
4992

Sample Mismatched Records:


,CustomerId,HasCrCard_account,HasCrCard_reference
1,15647311,1,0
2,15619304,0,1
5,15574012,0,1
7,15656148,0,1
8,15792365,1,0
11,15737173,0,1
12,15632264,0,1
15,15643966,1,0
16,15737452,0,1
21,15597945,0,1


In [53]:

# Restore correct HasCrCard values using CustomerId

card_lookup = reference_df.set_index("CustomerId")["HasCrCard"]

account_clean["HasCrCard"] = (
    account_clean["CustomerId"]
    .map(card_lookup)
    .astype("int64")
)

# Check corrected distribution
print("Corrected HasCrCard Distribution:")
print(account_clean["HasCrCard"].value_counts())

# Validate corrected values against reference
assert account_clean["HasCrCard"].notna().all()

assert set(account_clean["HasCrCard"].unique()) == {0, 1}

assert (
    account_clean["HasCrCard"].to_numpy()
    ==
    account_clean["CustomerId"].map(card_lookup).to_numpy()
).all()

assert len(account_clean) == 10000

print("\nHasCrCard correction validation passed!")


Corrected HasCrCard Distribution:
HasCrCard
1    7055
0    2945
Name: count, dtype: int64

HasCrCard correction validation passed!


### 3.12A HasCrCard — Reference Data Correction

**Issue identified:**
- The HasCrCard column initially contained 5,151 Yes values and 4,849 No values.
- Its values were identical to IsActiveMember for all 10,000 customers.
- Comparison with the clean reference dataset identified 4,992 incorrect HasCrCard records.

**Correction performed:**
- Matched account records with the clean reference dataset using CustomerId.
- Restored the correct HasCrCard values for all customers.
- Retained the binary encoding: 1 = Has Credit Card, 0 = No Credit Card.

**Final validation results:**
- Customers with credit cards: 7,055 (70.55%)
- Customers without credit cards: 2,945 (29.45%)
- Corrected records: 4,992
- Missing values: 0
- Total records preserved: 10,000

**Business impact:** Correcting this column prevents inaccurate credit-card ownership statistics and misleading churn analysis.

**Result:** HasCrCard data quality issue resolved successfully.

### 3.13 Exited (Customer Churn) Validation

**Objective:** Validate the target variable used to identify customers who have left the bank.

**Business importance:** Exited is the primary outcome variable for this project. Incorrect values could distort churn rates, customer segmentation, and retention insights.

**Validation approach:**
1. Inspect the data type.
2. Check for missing values.
3. Review the distribution of churned and retained customers.
4. Confirm that only valid binary values (0 and 1) are present.
5. Compare the values against the reference dataset if necessary.

**Business definition:**
- 1 = Customer exited (churned)
- 0 = Customer retained

In [54]:

# Inspect Exited column

print("Exited Data Type:")
print(account_clean["Exited"].dtype)

print("\nMissing Values:")
print(account_clean["Exited"].isna().sum())

print("\nUnique Values:")
print(account_clean["Exited"].unique())

print("\nExited Distribution:")
print(
    account_clean["Exited"]
    .value_counts(dropna=False)
)

print("\nTotal Records:")
print(len(account_clean))


Exited Data Type:
int64

Missing Values:
0

Unique Values:
[1 0]

Exited Distribution:
Exited
0    7963
1    2037
Name: count, dtype: int64

Total Records:
10000


In [55]:

# Validate Exited against the clean reference dataset

exited_check = account_clean[
    ["CustomerId", "Exited"]
].merge(
    reference_df[["CustomerId", "Exited"]],
    on="CustomerId",
    how="left",
    suffixes=("_account", "_reference"),
    validate="one_to_one"
)

# Check for missing reference matches
missing_matches = (
    exited_check["Exited_reference"].isna().sum()
)

# Compare churn values
exited_mismatch = exited_check[
    exited_check["Exited_account"] !=
    exited_check["Exited_reference"]
]

print("Total Records Compared:", len(exited_check))

print("\nMissing Reference Matches:", missing_matches)

print("\nExited Mismatch Count:", len(exited_mismatch))

print("\nSample Mismatched Records:")
display(exited_mismatch.head(10))


Total Records Compared: 10000

Missing Reference Matches: 0

Exited Mismatch Count: 0

Sample Mismatched Records:


,CustomerId,Exited_account,Exited_reference


### 3.13 Exited (Customer Churn) Validation — Summary

**Findings:**
- Exited contains 10,000 records.
- Data type: int64.
- No missing values were identified.
- Only valid binary values (0 and 1) are present.

**Churn distribution:**
- Retained customers (Exited = 0): 7,963 (79.63%)
- Churned customers (Exited = 1): 2,037 (20.37%)

**Reference validation:**
- Compared all records against the clean reference dataset using CustomerId.
- Missing reference matches: 0
- Exited mismatches: 0

**Cleaning decision:** No transformation or correction was required.

**Business observation:** Approximately 20.37% of customers have churned. This will serve as the baseline churn rate for subsequent customer segmentation analysis.

**Result:** Exited validation completed successfully.

### 3.14 Merging Customer and Account Datasets

**Objective:** Combine the cleaned customer and account datasets into one analytical table using CustomerId.

**Business importance:** A unified dataset enables analysis of customer churn across demographics, financial characteristics, and account activity.

**Merge strategy:**
- Join key: CustomerId
- Join type: Inner Join
- Relationship: One-to-One
- Remove the redundant Tenure column from the account dataset because both tables have already been verified to contain identical tenure values.

**Expected output:**
- 10,000 customer records
- 13 columns
- One record per CustomerId

**Validation:** Verify row count, column count, CustomerId uniqueness, and missing values after merging.

In [57]:

# Merge cleaned customer and account datasets

bank_churn_clean = pd.merge(
    customer_clean,
    account_clean.drop(columns=["Tenure"]),
    on="CustomerId",
    how="inner",
    validate="one_to_one"
)

# Inspect merged dataset
print("Merged Dataset Shape:")
print(bank_churn_clean.shape)

print("\nMerged Column Names:")
print(bank_churn_clean.columns.tolist())

print("\nFirst 5 Records:")
display(bank_churn_clean.head())


Merged Dataset Shape:
(10000, 13)

Merged Column Names:
['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'EstimatedSalary', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Exited']

First 5 Records:


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,EstimatedSalary,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
0,15634602,Hargrave,619,France,Female,42,2,101348.88,0.0,1,1,1,1
1,15647311,Hill,608,Spain,Female,41,1,112542.58,83807.86,1,0,1,0
2,15619304,Onio,502,France,Female,42,8,113931.57,159660.8,3,1,0,1
3,15701354,Boni,699,France,Female,39,1,93826.63,0.0,2,0,0,0
4,15737888,Mitchell,850,Spain,Female,43,2,79084.1,125510.82,1,1,1,0


In [58]:

# Final merged dataset validation

print("Dataset Shape:")
print(bank_churn_clean.shape)

print("\nDuplicate CustomerIds:")
print(bank_churn_clean["CustomerId"].duplicated().sum())

print("\nMissing Values by Column:")
print(bank_churn_clean.isna().sum())

print("\nTotal Missing Values:")
print(bank_churn_clean.isna().sum().sum())

print("\nColumn Data Types:")
print(bank_churn_clean.dtypes)

# Final assertions
assert bank_churn_clean.shape == (10000, 13)
assert bank_churn_clean["CustomerId"].is_unique
assert bank_churn_clean.isna().sum().sum() == 0

print("\nFinal merge validation passed!")


Dataset Shape:
(10000, 13)

Duplicate CustomerIds:
0

Missing Values by Column:
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
EstimatedSalary    0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
Exited             0
dtype: int64

Total Missing Values:
0

Column Data Types:
CustomerId           int64
Surname                str
CreditScore          int64
Geography              str
Gender                 str
Age                  int64
Tenure               int64
EstimatedSalary    Float64
Balance            Float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
Exited               int64
dtype: object

Final merge validation passed!


### 3.14 Dataset Merging — Summary

**Objective:** Combine cleaned customer and account information into one analytical dataset.

**Merge performed:**
- Joined customer_clean and account_clean using CustomerId.
- Used an inner join with one-to-one relationship validation.
- Removed the redundant Tenure column from the account dataset.
- Created the final DataFrame: bank_churn_clean.

**Final validation results:**
- Total records: 10,000
- Total columns: 13
- Duplicate CustomerIds: 0
- Missing values: 0
- All CustomerIds are unique.
- Numeric and categorical columns have appropriate data types.

**Result:** Successfully created a unified, validated Bank Customer Churn dataset ready for further analysis and reporting.

### 3.15 Exporting the Cleaned Dataset

**Objective:** Export the final validated Bank Customer Churn dataset into CSV and Excel formats.

**Output files:**
- Data/Cleaned/bank_churn_clean.csv
- Data/Cleaned/bank_churn_clean.xlsx

**Purpose:**
- CSV: PostgreSQL, Python EDA, and Power BI.
- Excel: PivotTables, business analysis, and manual data review.

**Expected output:**
- 10,000 rows
- 13 columns
- No missing values
- No duplicate CustomerIds

Both files will contain the same cleaned dataset.

In [59]:

# Export cleaned dataset to CSV and Excel

from pathlib import Path

project_root = Path(r"E:\BA Projects\Bank_Customer_Churn_Analysis")

# Create output folder
cleaned_path = project_root / "Data" / "Cleaned"
cleaned_path.mkdir(parents=True, exist_ok=True)

# Define output files
csv_file = cleaned_path / "bank_churn_clean.csv"
excel_file = cleaned_path / "bank_churn_clean.xlsx"

# Export CSV
bank_churn_clean.to_csv(
    csv_file,
    index=False,
    encoding="utf-8"
)

# Export Excel
bank_churn_clean.to_excel(
    excel_file,
    index=False,
    sheet_name="Bank_Churn_Cleaned",
    engine="openpyxl"
)

print("CSV exported successfully:", csv_file)
print("Excel exported successfully:", excel_file)

print("\nTotal Rows:", len(bank_churn_clean))
print("Total Columns:", len(bank_churn_clean.columns))


CSV exported successfully: E:\BA Projects\Bank_Customer_Churn_Analysis\Data\Cleaned\bank_churn_clean.csv
Excel exported successfully: E:\BA Projects\Bank_Customer_Churn_Analysis\Data\Cleaned\bank_churn_clean.xlsx

Total Rows: 10000
Total Columns: 13


### 3.16 Exported Dataset Validation

**Objective:** Verify that the exported CSV and Excel files contain the same cleaned data as the original Pandas DataFrame.

**Business importance:** Export validation ensures that no records, columns, or values were lost or modified during file creation.

**Validation approach:**
1. Read both exported files into Pandas.
2. Compare their row and column counts.
3. Check for missing values and duplicate CustomerIds.
4. Compare the exported data against the cleaned DataFrame.
5. Confirm that both files are ready for downstream analysis.

In [60]:

# Read exported CSV and Excel files

csv_check = pd.read_csv(csv_file)

excel_check = pd.read_excel(excel_file)

# Check dataset shapes
print("Original DataFrame:", bank_churn_clean.shape)
print("CSV File:", csv_check.shape)
print("Excel File:", excel_check.shape)

# Check missing values
print("\nMissing Values:")
print("CSV:", csv_check.isna().sum().sum())
print("Excel:", excel_check.isna().sum().sum())

# Check duplicate CustomerIds
print("\nDuplicate CustomerIds:")
print("CSV:", csv_check["CustomerId"].duplicated().sum())
print("Excel:", excel_check["CustomerId"].duplicated().sum())

# Validate dataset structure
assert csv_check.shape == (10000, 13)
assert excel_check.shape == (10000, 13)

assert csv_check.columns.tolist() == bank_churn_clean.columns.tolist()
assert excel_check.columns.tolist() == bank_churn_clean.columns.tolist()

assert csv_check["CustomerId"].is_unique
assert excel_check["CustomerId"].is_unique

assert csv_check.isna().sum().sum() == 0
assert excel_check.isna().sum().sum() == 0

print("\nBasic export validation passed!")


Original DataFrame: (10000, 13)
CSV File: (10000, 13)
Excel File: (10000, 13)

Missing Values:
CSV: 0
Excel: 0

Duplicate CustomerIds:
CSV: 0
Excel: 0

Basic export validation passed!


## 4. Data Cleaning — Final Summary

### Project: Bank Customer Churn Analysis

**Objective:** Prepare a clean, consistent, and validated customer churn dataset for SQL analysis, exploratory data analysis, and Power BI reporting.

### Data Cleaning Activities Completed

1. Removed duplicate customer and account records.
2. Handled missing and invalid customer information using the reference dataset.
3. Standardized Geography and Gender categories.
4. Converted EstimatedSalary and Balance into numeric formats.
5. Validated CreditScore, Age, Tenure, and NumOfProducts.
6. Corrected HasCrCard values using the reference dataset.
7. Standardized IsActiveMember and Exited into binary indicators.
8. Merged customer and account datasets using CustomerId.
9. Exported the cleaned dataset into CSV and Excel formats.
10. Validated exported dataset structure, completeness, and accuracy.

### Final Dataset Quality

| Metric | Result |
|---|---|
| Total Customers | 10,000 |
| Total Columns | 13 |
| Duplicate CustomerIds | 0 |
| Missing Values | 0 |
| Churned Customers | 2,037 |
| Retained Customers | 7,963 |
| Overall Churn Rate | 20.37% |

### Final Deliverables

- `Data/Cleaned/bank_churn_clean.csv`
- `Data/Cleaned/bank_churn_clean.xlsx`
- `Python/01_Data_Exploration.ipynb`
- `Python/02_Data_Cleaning.ipynb`

### Conclusion

The data preparation phase is complete. The cleaned dataset is ready for PostgreSQL analysis, Python exploratory data analysis, and Power BI dashboard development.